# Juntar as tabelas do INE

Junta os CSV do INE (e o da AIMA) num só ficheiro: `data/T06_merged_data.csv`.

## 1. Bibliotecas

In [ ]:
library(tidyverse)

## 2. Funções para ler os ficheiros do INE

In [ ]:
ler_linhas_ine <- function(ficheiro) {
  todas_linhas <- read_lines(ficheiro, locale = locale(encoding = "latin1"))

  # só as linhas que começam com um código de região ("PT: ", "11A: ", ...)
  e_dados <- str_detect(todas_linhas, "^(\\d{4})?;?[0-9A-Z]{1,4}: ")

  read_delim(
    I(todas_linhas[e_dados]),
    delim = ";",
    col_names = FALSE,
    col_types = cols(.default = col_character()),
    show_col_types = FALSE
  )
}

# vírgula decimal -> número
para_numero <- function(x) round(parse_number(x, locale = locale(decimal_mark = ",")), 2)

## 3. Regiões (NUTS III e ilhas)

In [ ]:
regioes <- ler_linhas_ine("data/T06_homes_completed.csv") |>
  select(X1) |>
  separate(X1, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  filter(nchar(nuts_code) == 3 | nuts_code %in% c("20", "30"))

regioes

## 4. Tabelas com um valor por ano

O `passo` é 5 nas licenças e nos preços de venda (cada ano tem 5 colunas e ficamos com a primeira, o total).

In [ ]:
ler_tabela_anual <- function(ficheiro, anos, nome_coluna, passo = 1) {
  colunas_dos_anos <- 2 + (seq_along(anos) - 1) * passo

  ler_linhas_ine(ficheiro) |>
    select(all_of(c(1, colunas_dos_anos))) |>
    set_names(c("region_raw", anos)) |>
    separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
    pivot_longer(-c(nuts_code, region), names_to = "year", values_to = "valor") |>
    mutate(year = as.integer(year), valor = para_numero(valor)) |>
    select(nuts_code, year, valor) |>
    set_names(c("nuts_code", "year", nome_coluna))
}

salario      <- ler_tabela_anual("data/T06_average_wage.csv",      2024:2021, "wage_monthly")
rendimento   <- ler_tabela_anual("data/T06_median_income.csv",     2024:2018, "income_median")
renda        <- ler_tabela_anual("data/T06_median_rent_m2.csv",    2025:2020, "rent_m2")
casas        <- ler_tabela_anual("data/T06_homes_completed.csv",   2025:2011, "homes_completed")
estrangeiros <- ler_tabela_anual("data/T06_foreign_residents.csv", 2023:2021, "foreign_residents")
migracao     <- ler_tabela_anual("data/T06_net_migration.csv",     2025:2021, "net_migration")

licencas <- ler_tabela_anual("data/T06_homes_licensed.csv", 2025:2002, "homes_licensed", passo = 5)
vendas   <- ler_tabela_anual("data/T06_sale_price_m2.csv",  2025:2019, "sale_price_m2",   passo = 5)

# os Açores aparecem duas vezes na renda (20 e 200)
renda <- renda |> filter(nuts_code != "200")

salario
renda

## 5. População por idade

In [ ]:
nomes_idades <- c("total", "a0_4", "a5_9", "a10_14", "a15_19", "a20_24", "a25_29", "a30_34",
                  "a35_39", "a40_44", "a45_49", "a50_54", "a55_59", "a60_64", "a65_69",
                  "a70_74", "a75_79", "a80_84", "a85_mais")

populacao_idades <- ler_linhas_ine("data/T06_population_by_age.csv") |>
  mutate(year = as.integer(X1)) |>
  fill(year) |>   # o ano só vem na primeira linha de cada bloco
  select(year, region_raw = X2, X3:X21) |>
  set_names(c("year", "region_raw", nomes_idades)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  mutate(across(total:a85_mais, para_numero))

populacao <- populacao_idades |>
  transmute(
    nuts_code, year,
    population   = total,
    share_0_14   = (a0_4 + a5_9 + a10_14) / total * 100,
    share_15_24  = (a15_19 + a20_24) / total * 100,
    share_25_34  = (a25_29 + a30_34) / total * 100,
    share_35_44  = (a35_39 + a40_44) / total * 100,
    share_45_54  = (a45_49 + a50_54) / total * 100,
    share_55_64  = (a55_59 + a60_64) / total * 100,
    share_65_79  = (a65_69 + a70_74 + a75_79) / total * 100,
    share_80plus = (a80_84 + a85_mais) / total * 100,
    share_20_34  = (a20_24 + a25_29 + a30_34) / total * 100
  ) |>
  mutate(across(starts_with("share_"), ~ round(.x, 2)))

populacao

## 6. Censos 2021 e parque habitacional 2022

In [ ]:
# colunas: total, propriedade, arrendamento, outra situação
regime <- ler_linhas_ine("data/T06_households_tenure_2021.csv") |>
  select(region_raw = X1, households_2021 = X2, rented_households_2021 = X4) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  mutate(across(c(households_2021, rented_households_2021), para_numero)) |>
  select(nuts_code, households_2021, rented_households_2021)

parque <- ler_linhas_ine("data/T06_dwelling_stock_2022.csv") |>
  select(region_raw = X1, dwellings_2022 = X2) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  mutate(dwellings_2022 = para_numero(dwellings_2022)) |>
  select(nuts_code, dwellings_2022)

regime
parque

## 7. Dormidas (de mensal para anual)

Somamos os meses de cada ano e ficamos com 2022 a 2025.

In [ ]:
# linha dos meses: ";Agosto de 2026;Julho de 2026;..."
linhas_turismo <- read_lines("data/T06_tourist_nights.csv", locale = locale(encoding = "latin1"))
linha_meses    <- linhas_turismo[str_detect(linhas_turismo, "^;[^;]+ de \\d{4};")][1]

nomes_meses <- str_split(linha_meses, ";")[[1]]
nomes_meses <- nomes_meses[nomes_meses != ""]

dormidas_mensais <- ler_linhas_ine("data/T06_tourist_nights.csv") |>
  select(1:(length(nomes_meses) + 1)) |>
  set_names(c("region_raw", nomes_meses)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  pivot_longer(-c(nuts_code, region), names_to = "month", values_to = "tourist_nights") |>
  mutate(
    year = as.integer(str_extract(month, "\\d{4}")),
    tourist_nights = para_numero(tourist_nights)
  )

dormidas <- dormidas_mensais |>
  inner_join(regioes |> select(nuts_code), by = "nuts_code") |>
  filter(year >= 2022, year <= 2025) |>
  group_by(nuts_code, year) |>
  summarise(tourist_nights = sum(tourist_nights), .groups = "drop")

dormidas

## 8. Estrangeiros residentes em 2024 (AIMA)

Este ficheiro só tem 2024 e está em UTF-8, por isso lemos com `read_csv`.

In [ ]:
estrangeiros_aima <- read_csv(
  "data/T06_foreign_residents_aima_2024.csv",
  col_types = cols(nuts_code = col_character()),
  show_col_types = FALSE
) |>
  select(nuts_code, year, foreign_residents_aima_2024)

# 26 regiões, sem repetidas e iguais às da nossa lista
stopifnot(
  nrow(estrangeiros_aima) == 26,
  !anyDuplicated(estrangeiros_aima$nuts_code),
  all(estrangeiros_aima$year == 2024),
  setequal(estrangeiros_aima$nuts_code, regioes$nuts_code)
)

estrangeiros_aima

## 9. Juntar tudo

Onde não há dados fica `NA`.

In [ ]:
esqueleto <- expand_grid(regioes, year = 2021:2024)

unido <- esqueleto |>
  left_join(salario,       by = c("nuts_code", "year")) |>
  left_join(rendimento,    by = c("nuts_code", "year")) |>
  left_join(renda,         by = c("nuts_code", "year")) |>
  left_join(vendas,        by = c("nuts_code", "year")) |>
  left_join(casas,         by = c("nuts_code", "year")) |>
  left_join(licencas,      by = c("nuts_code", "year")) |>
  left_join(estrangeiros,  by = c("nuts_code", "year")) |>
  left_join(populacao,     by = c("nuts_code", "year")) |>
  left_join(migracao,      by = c("nuts_code", "year")) |>
  left_join(dormidas,      by = c("nuts_code", "year")) |>
  left_join(regime,        by = "nuts_code") |>   # repetido em todos os anos
  left_join(parque,        by = "nuts_code") |>   # repetido em todos os anos
  left_join(estrangeiros_aima, by = c("nuts_code", "year")) |>
  arrange(nuts_code, year)

glimpse(unido)

## 10. Verificações

Devem ser 104 linhas, 26 regiões e 0 linhas repetidas.

In [ ]:
nrow(unido)
n_distinct(unido$nuts_code)
unido |> count(nuts_code, year) |> filter(n > 1) |> nrow()

unido |>
  summarise(across(everything(), ~ sum(is.na(.x)))) |>
  pivot_longer(everything(), names_to = "column", values_to = "missing")

Cada tabela tem as 26 regiões? Tem de dar 0 em todas.

In [ ]:
tabelas <- list(
  salario = salario, rendimento = rendimento, renda = renda, vendas = vendas,
  licencas = licencas, estrangeiros = estrangeiros, populacao = populacao,
  migracao = migracao, regime = regime, parque = parque, dormidas = dormidas,
  estrangeiros_aima = estrangeiros_aima
)

for (nome in names(tabelas)) {
  regioes_em_falta <- setdiff(regioes$nuts_code, tabelas[[nome]]$nuts_code)
  cat(nome, ":", length(regioes_em_falta), "regiões em falta\n")
}

# 4 anos de turismo por região (2022-2025)
dormidas |> count(nuts_code) |> filter(n != 4)

Comparação com os ficheiros originais, em 2024.

In [ ]:
# 11A: salário 1620.1, renda 9.14, venda 1986, 5804 casas
unido |>
  filter(nuts_code == "11A", year == 2024) |>
  select(nuts_code, year, wage_monthly, rent_m2, sale_price_m2, homes_completed)

# 1A0: população 2419228, migração 56494, 1072111 alojamentos, 31.4% a arrendar, AIMA 574895
# 150: população 577095, AIMA 167321
unido |>
  filter(nuts_code %in% c("1A0", "150"), year == 2024) |>
  transmute(nuts_code, population, net_migration, dwellings_2022,
            rented_share = round(rented_households_2021 / households_2021 * 100, 1),
            foreign_residents_aima_2024)

### O ranking muda com o rendimento mediano?

Comparamos a ordem das regiões com a correlação de Spearman. Perto de 1 quer dizer que é a mesma.

In [ ]:
esforco <- unido |>
  filter(year %in% c(2021, 2024)) |>
  transmute(
    nuts_code, year,
    esforco_salario    = rent_m2 * 80 / wage_monthly * 100,
    esforco_rendimento = rent_m2 * 80 * 12 / income_median * 100
  ) |>
  pivot_wider(names_from = year, values_from = c(esforco_salario, esforco_rendimento))

# ordem em 2024
cor(esforco$esforco_salario_2024, esforco$esforco_rendimento_2024, method = "spearman")

# regiões que pioraram entre 2021 e 2024
cor(esforco$esforco_salario_2024 - esforco$esforco_salario_2021,
    esforco$esforco_rendimento_2024 - esforco$esforco_rendimento_2021, method = "spearman")

## 11. Guardar

In [ ]:
write_csv(unido, "data/T06_merged_data.csv")